<a href="https://colab.research.google.com/github/GiaPhu-1511/RT-SWT-005-nhom6/blob/main/SWT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL CÀI ĐẶT SẠCH - CHỈ CHẠY 1 LẦN SAU KHI RESTART]
!pip install -U google-genai pandas requests
print("--- Đã cài đặt xong thư viện sạch ---")

--- Đã cài đặt xong thư viện sạch ---


In [ ]:
# [CELL CẦN CỨU CÁNH - DUMP TOÀN BỘ METADATA CỦA MODEL]
!pip install -q -U google-cloud-aiplatform

import os
from google.cloud import aiplatform

# 1. Đặt đường dẫn key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"

# 2. Khởi tạo
PROJECT_ID = "rbl-core-2026"
LOCATION = "us-central1"

print(f"[INFO] Đang truy vấn trực tiếp model trên GCP...")

try:
    aiplatform.init(project=PROJECT_ID, location=LOCATION)

    # Liệt kê tất cả các model khả dụng trong dự án
    models = aiplatform.Model.list()
    print(f"\n[DANH SÁCH MODEL TÌM THẤY TRÊN DỰ ÁN {PROJECT_ID}]:")
    for m in models:
        print(f" -> Name: {m.name}, Resource Name: {m.resource_name}")

    # Thử lấy chi tiết một model
    print("\n[ĐANG THỬ LẤY MODEL GEMINI-1.5-FLASH]:")
    # Đường dẫn đầy đủ chuẩn của Vertex AI
    model_name = "publishers/google/models/gemini-1.5-flash-001"
    model = aiplatform.GenerativeModel(model_name)
    print(" -> SUCCESS: Đã tìm thấy model!")

except Exception as e:
    print(f"\n[LỖI CHI TIẾT TỪ GOOGLE]:\n{e}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.10.0 which is incompatible.
[INFO] Đang truy vấn trực tiếp model trên GCP...

[DANH SÁCH MODEL TÌM THẤY TRÊN DỰ ÁN rbl-core-2026]:

[ĐANG THỬ LẤY MODEL GEMINI-1.5-FLASH]:

[LỖI CHI TIẾT TỪ GOOGLE]:
module 'google.cloud.aiplatform' has no attribute 'GenerativeModel'


In [ ]:
# [CELL 1 - XÁC THỰC HẠ TẦNG]
!gcloud auth activate-service-account --key-file=/content/new-gcp-key.json
!gcloud config set project rbl-core-2026
!gcloud services list --enabled --filter="name:aiplatform"

Activated service account credentials for: [rbl-worker@rbl-core-2026.iam.gserviceaccount.com]
- '@type': type.googleapis.com/google.rpc.ErrorInfo
  domain: googleapis.com
  metadata:
    activationUrl: https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=913058957673
    consumer: projects/913058957673
    containerInfo: '913058957673'
    service: cloudresourcemanager.googleapis.com
    serviceTitle: Cloud Resource Manager API
  reason: SERVICE_DISABLED
- '@type': type.googleapis.com/google.rpc.LocalizedMessage
  locale: en-US
  message: Cloud Resource Manager API has not been used in project 913058957673 before
    or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=913058957673
    then retry. If you enabled this API recently, wait a few minutes for the action
    to propagate to our systems and retry.
- '@type': type.googleapis.com/google.rpc.Help


In [ ]:
# [CELL 2 - QUÉT MODEL BẰNG GOOGLE-GENAI SDK]
import os
from google import genai
from google.genai import types

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"

client = genai.Client(
    vertexai=True,
    project="rbl-core-2026",
    location="us-central1"
)

print("\n--- CÁC MODEL KHẢ DỤNG TRÊN DỰ ÁN CỦA SẾP ---")
try:
    for m in client.models.list():
        print(f"Model ID: {m.name}")
except Exception as e:
    print(f"Lỗi: {e}")


--- CÁC MODEL KHẢ DỤNG TRÊN DỰ ÁN CỦA SẾP ---
Model ID: publishers/google/models/imageclassification-efficientnet
Model ID: publishers/google/models/occupancy-analytics
Model ID: publishers/google/models/multimodalembedding
Model ID: publishers/google/models/pt-test
Model ID: publishers/google/models/imageclassification-vit
Model ID: publishers/google/models/bert-base
Model ID: publishers/google/models/vehicle-detector
Model ID: publishers/google/models/language-v1-classify-text-v1
Model ID: publishers/google/models/language-v1-analyze-sentiment
Model ID: publishers/google/models/language-v1-analyze-entity-sentiment
Model ID: publishers/google/models/language-v1-analyze-syntax
Model ID: publishers/google/models/resnet50
Model ID: publishers/google/models/imagesegmentation-deeplabv3
Model ID: publishers/google/models/imageobjectdetection-yolo
Model ID: publishers/google/models/owlvit-base-patch32
Model ID: publishers/google/models/object-detector
Model ID: publishers/google/models/ppe-

In [ ]:
# [CELL: ENTERPRISE TEST ENGINE - MÔ PHỎNG BÀI BÁO]
import os
import json
import pandas as pd
import re
from google import genai

# Setup
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"
client = genai.Client(vertexai=True, project="rbl-core-2026", location="asia-southeast1")
TARGET_MODEL = "gemini-3.5-flash"
TARGET_BASE_URL = "https://disease.sh"

def get_llm_response(prompt):
    response = client.models.generate_content(model=TARGET_MODEL, contents=prompt)
    return response.text

def run_fuzzing_experiment(endpoints):
    results = []

    for ep in endpoints:
        print(f"\n>>> Test Endpoint: {ep}")

        # 1. Nhánh A: LLM Only (Initial Prompt)
        prompt_a = f"Task: Generate valid JSON body or query params for GET/POST {ep} on {TARGET_BASE_URL}. Output ONLY JSON."
        params_a = get_llm_response(prompt_a)

        # 2. Nhánh B: LLM + Feedback Loop [cite: 261]
        # Giả lập: nếu test A lỗi -> sinh Feedback Prompt
        # Ở đây ta thực thi request thực tế
        import requests
        try:
            # Test nhánh A
            resp_a = requests.get(f"{TARGET_BASE_URL}{ep}", timeout=5)
            status_a = resp_a.status_code

            if status_a >= 400:
                print(f"  -> Nhánh A Lỗi ({status_a}), kích hoạt Feedback Loop...")
                feedback_prompt = f"API returned error: {resp_a.text}. Refine input parameters for {ep}."
                params_b = get_llm_response(feedback_prompt)
                resp_b = requests.get(f"{TARGET_BASE_URL}{ep}", timeout=5)
                status_b = resp_b.status_code
            else:
                status_b = status_a
                params_b = params_a

            results.append({"endpoint": ep, "method_a_status": status_a, "method_b_status": status_b})
            print(f"  -> Kết quả: Nhánh A={status_a}, Nhánh B={status_b}")

        except Exception as e:
            print(f"  -> Lỗi kết nối: {e}")

    return pd.DataFrame(results)

# Danh sách endpoint từ OpenAPI mẫu
endpoints = ["/v3/covid-19/all", "/v3/covid-19/countries"]
df = run_fuzzing_experiment(endpoints)
df.to_csv('/content/drive/MyDrive/RBL-experiment/results/comparison_results.csv', index=False)
print("\n[FINISH] Đã so sánh xong 2 phương pháp!")


>>> Test Endpoint: /v3/covid-19/all
  -> Kết quả: Nhánh A=200, Nhánh B=200

>>> Test Endpoint: /v3/covid-19/countries
  -> Kết quả: Nhánh A=200, Nhánh B=200

[FINISH] Đã so sánh xong 2 phương pháp!


In [ ]:
# [CELL: disease.sh - LLM-only vs LLM+Feedback]
import os, re, json, requests, pandas as pd
from google import genai

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"
client = genai.Client(vertexai=True, project="rbl-core-2026", location="asia-southeast1")
TARGET_MODEL = "gemini-3.5-flash"
BASE_URL = "https://disease.sh"
SERVICE_NAME = "disease.sh"

ENDPOINTS = [
    # --- COVID-19: Worldometers ---
    {"method": "GET", "path": "/v3/covid-19/all", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/countries", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/countries/{country}", "path_params": {"country": "tên hoặc mã ISO 3166 của quốc gia, ví dụ USA"}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/states", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/states/{states}", "path_params": {"states": "tên một bang của Mỹ, ví dụ California"}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/continents", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/continents/{continent}", "path_params": {"continent": "tên một châu lục, ví dụ Asia"}, "body_params": {}},

    # --- COVID-19: JHUCSSE ---
    {"method": "GET", "path": "/v3/covid-19/jhucsse", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/jhucsse/counties", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/jhucsse/counties/{county}", "path_params": {"county": "tên một county của Mỹ, ví dụ Los Angeles"}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/historical", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/historical/all", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/historical/{country}", "path_params": {"country": "tên hoặc mã quốc gia hợp lệ"}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/historical/{country}/{province}", "path_params": {"country": "tên quốc gia hợp lệ, ví dụ USA", "province": "tên một tỉnh/bang thuộc quốc gia đó, ví dụ California"}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/historical/usacounties", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/historical/usacounties/{state}", "path_params": {"state": "tên một bang của Mỹ hợp lệ, ví dụ Texas"}, "body_params": {}},

    # --- COVID-19: NYT ---
    {"method": "GET", "path": "/v3/covid-19/nyt/states", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/nyt/states/{state}", "path_params": {"state": "tên một bang của Mỹ, ví dụ Illinois"}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/nyt/usa", "path_params": {}, "body_params": {}},

    # --- COVID-19: Apple mobility ---
    {"method": "GET", "path": "/v3/covid-19/apple/mobility/{subregion}", "path_params": {"subregion": "tên một quốc gia hoặc thành phố hợp lệ trong dữ liệu Apple Mobility, ví dụ United States"}, "body_params": {}},

    # --- COVID-19: Government ---
    {"method": "GET", "path": "/v3/gov", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/gov/{iso2}", "path_params": {"iso2": "mã ISO2 của một quốc gia có dữ liệu chính phủ, ví dụ FR"}, "body_params": {}},

    # --- COVID-19: Vaccine ---
    {"method": "GET", "path": "/v3/covid-19/vaccine", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/vaccine/coverage", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/vaccine/coverage/countries", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/vaccine/coverage/countries/{country}", "path_params": {"country": "tên quốc gia hợp lệ, ví dụ Germany"}, "body_params": {}},

    # --- COVID-19: Therapeutics ---
    {"method": "GET", "path": "/v3/covid-19/therapeutics/trials", "path_params": {}, "body_params": {}},

    # --- COVID-19: Variants ---
    {"method": "GET", "path": "/v3/covid-19/variants", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/covid-19/variants/countries", "path_params": {}, "body_params": {}},

    # --- Influenza: CDC ---
    {"method": "GET", "path": "/v3/influenza/cdc", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v3/influenza/usailiweekly", "path_params": {}, "body_params": {}},
]
def get_llm_response(prompt):
    resp = client.models.generate_content(model=TARGET_MODEL, contents=prompt)
    return resp.text

def extract_answer(raw):
    m = re.search(r"<answer>(.*?)(?:</answer>|<answer>|$)", raw, re.S | re.I)
    return m.group(1).strip() if m else raw.strip()

def gen_value(api_path, pname, pdesc, error_feedback=None):
    if error_feedback:
        prompt = f"""You are an API tester. API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Previous attempt failed with error: {error_feedback}
Give me a corrected valid value. Output ONLY <answer>your answer<answer>"""
    else:
        prompt = f"""You are an API tester. Generate a valid value for the parameter below.
API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Output ONLY <answer>your answer<answer>"""
    return extract_answer(get_llm_response(prompt))

def build_call(ep, feedback=None):
    path = ep["path"]
    body = {}
    for pname, pdesc in ep["path_params"].items():
        fb = feedback.get(pname) if feedback else None
        val = gen_value(ep["path"], pname, pdesc, fb)
        path = path.replace("{" + pname + "}", str(val))
    for pname, pdesc in ep["body_params"].items():
        fb = feedback.get(pname) if feedback else None
        body[pname] = gen_value(ep["path"], pname, pdesc, fb)
    return path, body

def run_endpoint(ep):
    path_a, body_a = build_call(ep)
    try:
        resp_a = requests.request(ep["method"], BASE_URL + path_a, json=body_a or None, timeout=10)
        status_a = resp_a.status_code
        err_text = resp_a.text[:300]
    except Exception as e:
        status_a, err_text = -1, str(e)

    status_b = status_a
    if status_a >= 400 or status_a == -1:
        feedback = {k: err_text for k in list(ep["path_params"]) + list(ep["body_params"])}
        path_b, body_b = build_call(ep, feedback)
        try:
            resp_b = requests.request(ep["method"], BASE_URL + path_b, json=body_b or None, timeout=10)
            status_b = resp_b.status_code
        except Exception as e:
            status_b = -1
    return status_a, status_b

results = []
for ep in ENDPOINTS:
    print(f">>> Test {ep['method']} {ep['path']}")
    sa, sb = run_endpoint(ep)
    print(f"    w/o feedback={sa}  w/ feedback={sb}")
    results.append({"service": SERVICE_NAME, "endpoint": ep["path"], "status_wo_feedback": sa, "status_w_feedback": sb})

df = pd.DataFrame(results)
def is2xx(s): return isinstance(s, int) and 200 <= s < 300
def is5xx(s): return isinstance(s, int) and 500 <= s < 600

summary = {
    "service": SERVICE_NAME,
    "2XX_wo_feedback": df["status_wo_feedback"].apply(is2xx).sum(),
    "2XX_w_feedback": df["status_w_feedback"].apply(is2xx).sum(),
    "5XX_wo_feedback": df["status_wo_feedback"].apply(is5xx).sum(),
    "5XX_w_feedback": df["status_w_feedback"].apply(is5xx).sum(),
}
print("\n[SUMMARY]", summary)

os.makedirs("/content/drive/MyDrive/RBL-experiment/results", exist_ok=True)
df.to_csv(f"/content/drive/MyDrive/RBL-experiment/results/{SERVICE_NAME}_results.csv", index=False)
print(f"\n[FINISH] Đã test xong {SERVICE_NAME}!")

>>> Test GET /v3/covid-19/all
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/countries
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/countries/{country}
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/states
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/states/{states}
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/continents
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/continents/{continent}
    w/o feedback=200  w/ feedback=200
>>> Test GET /v3/covid-19/jhucsse
    w/o feedback=502  w/ feedback=502
>>> Test GET /v3/covid-19/jhucsse/counties
    w/o feedback=502  w/ feedback=502
>>> Test GET /v3/covid-19/jhucsse/counties/{county}
    w/o feedback=502  w/ feedback=502
>>> Test GET /v3/covid-19/historical
    w/o feedback=502  w/ feedback=502
>>> Test GET /v3/covid-19/historical/all
    w/o feedback=502  w/ feedback=502
>>> Test GET /v3/covid-19/historical/{country}
    w/o feedback=

In [ ]:
# [CELL: languagetool - LLM-only vs LLM+Feedback]
import os, re, json, requests, pandas as pd
from google import genai

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"
client = genai.Client(vertexai=True, project="rbl-core-2026", location="asia-southeast1")
TARGET_MODEL = "gemini-3.5-flash"
BASE_URL = "https://api.languagetool.org"
SERVICE_NAME = "languagetool"

ENDPOINTS = [
    {"method": "POST", "path": "/v2/check", "path_params": {}, "body_params": {
        "text": "đoạn văn bản tiếng Anh cần kiểm tra ngữ pháp",
        "language": "mã ngôn ngữ, ví dụ en-US hoặc auto"
    }},
    {"method": "GET", "path": "/v2/languages", "path_params": {}, "body_params": {}},
    {"method": "GET", "path": "/v2/words", "path_params": {}, "body_params": {
        "username": "tên đăng nhập tài khoản LanguageTool",
        "apiKey": "API key của tài khoản LanguageTool"
    }},
    {"method": "POST", "path": "/v2/words/add", "path_params": {}, "body_params": {
        "word": "một từ tiếng Anh hợp lệ muốn thêm vào từ điển cá nhân",
        "username": "tên đăng nhập tài khoản LanguageTool",
        "apiKey": "API key của tài khoản LanguageTool"
    }},
    {"method": "POST", "path": "/v2/words/delete", "path_params": {}, "body_params": {
        "word": "một từ tiếng Anh hợp lệ muốn xoá khỏi từ điển cá nhân",
        "username": "tên đăng nhập tài khoản LanguageTool",
        "apiKey": "API key của tài khoản LanguageTool"
    }},
]
def get_llm_response(prompt):
    resp = client.models.generate_content(model=TARGET_MODEL, contents=prompt)
    return resp.text

def extract_answer(raw):
    m = re.search(r"<answer>(.*?)(?:</answer>|<answer>|$)", raw, re.S | re.I)
    return m.group(1).strip() if m else raw.strip()

def gen_value(api_path, pname, pdesc, error_feedback=None):
    if error_feedback:
        prompt = f"""You are an API tester. API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Previous attempt failed with error: {error_feedback}
Give me a corrected valid value. Output ONLY <answer>your answer<answer>"""
    else:
        prompt = f"""You are an API tester. Generate a valid value for the parameter below.
API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Output ONLY <answer>your answer<answer>"""
    return extract_answer(get_llm_response(prompt))

def build_call(ep, feedback=None):
    path = ep["path"]
    body = {}
    for pname, pdesc in ep["path_params"].items():
        fb = feedback.get(pname) if feedback else None
        val = gen_value(ep["path"], pname, pdesc, fb)
        path = path.replace("{" + pname + "}", str(val))
    for pname, pdesc in ep["body_params"].items():
        fb = feedback.get(pname) if feedback else None
        body[pname] = gen_value(ep["path"], pname, pdesc, fb)
    return path, body

def run_endpoint(ep):
    path_a, body_a = build_call(ep)
    try:
        if ep["method"] == "POST":
            # languagetool /v2/check chỉ nhận POST dạng form-data, không phải JSON
            resp_a = requests.post(BASE_URL + path_a, data=body_a, timeout=10)
        else:
            resp_a = requests.get(BASE_URL + path_a, timeout=10)
        status_a, err_text = resp_a.status_code, resp_a.text[:300]
    except Exception as e:
        status_a, err_text = -1, str(e)

    status_b = status_a
    if status_a >= 400 or status_a == -1:
        feedback = {k: err_text for k in list(ep["path_params"]) + list(ep["body_params"])}
        path_b, body_b = build_call(ep, feedback)
        try:
            if ep["method"] == "POST":
                resp_b = requests.post(BASE_URL + path_b, data=body_b, timeout=10)
            else:
                resp_b = requests.get(BASE_URL + path_b, timeout=10)
            status_b = resp_b.status_code
        except Exception:
            status_b = -1
    return status_a, status_b

results = []
for ep in ENDPOINTS:
    print(f">>> Test {ep['method']} {ep['path']}")
    sa, sb = run_endpoint(ep)
    print(f"    w/o feedback={sa}  w/ feedback={sb}")
    results.append({"service": SERVICE_NAME, "endpoint": ep["path"], "status_wo_feedback": sa, "status_w_feedback": sb})

df = pd.DataFrame(results)
def is2xx(s): return isinstance(s, int) and 200 <= s < 300
def is5xx(s): return isinstance(s, int) and 500 <= s < 600

summary = {
    "service": SERVICE_NAME,
    "2XX_wo_feedback": df["status_wo_feedback"].apply(is2xx).sum(),
    "2XX_w_feedback": df["status_w_feedback"].apply(is2xx).sum(),
    "5XX_wo_feedback": df["status_wo_feedback"].apply(is5xx).sum(),
    "5XX_w_feedback": df["status_w_feedback"].apply(is5xx).sum(),
}
print("\n[SUMMARY]", summary)

os.makedirs("/content/drive/MyDrive/RBL-experiment/results", exist_ok=True)
df.to_csv(f"/content/drive/MyDrive/RBL-experiment/results/{SERVICE_NAME}_results.csv", index=False)
print(f"\n[FINISH] Đã test xong {SERVICE_NAME}! (Lưu ý: giới hạn 20 req/phút/IP)")

>>> Test POST /v2/check
    w/o feedback=200  w/ feedback=200
>>> Test GET /v2/languages
    w/o feedback=200  w/ feedback=200
>>> Test GET /v2/words
    w/o feedback=400  w/ feedback=400
>>> Test POST /v2/words/add
    w/o feedback=400  w/ feedback=400
>>> Test POST /v2/words/delete
    w/o feedback=400  w/ feedback=400

[SUMMARY] {'service': 'languagetool', '2XX_wo_feedback': np.int64(2), '2XX_w_feedback': np.int64(2), '5XX_wo_feedback': np.int64(0), '5XX_w_feedback': np.int64(0)}

[FINISH] Đã test xong languagetool! (Lưu ý: giới hạn 20 req/phút/IP)


In [ ]:
# [CELL: genome-nexus - LLM-only vs LLM+Feedback]
import os, re, json, requests, pandas as pd
from google import genai

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"
client = genai.Client(vertexai=True, project="rbl-core-2026", location="asia-southeast1")
TARGET_MODEL = "gemini-3.5-flash"
BASE_URL = "https://www.genomenexus.org"
SERVICE_NAME = "genome-nexus"

# body_mode: "none" (không có body) | "array" (body là JSON array) | "object" (body là JSON object)
ENDPOINTS = [
    # --- annotation-controller ---
    {"method": "GET", "path": "/annotation/{variant}", "path_params": {"variant": "một biến thể HGVS hợp lệ, ví dụ 17:g.41242962_41242963insGA"}, "body_mode": "none"},
    {"method": "POST", "path": "/annotation", "path_params": {}, "body_mode": "array", "array_item_desc": "một biến thể HGVS hợp lệ, ví dụ 7:g.140453136A>T"},
    {"method": "POST", "path": "/annotation/dbsnp/", "path_params": {}, "body_mode": "array", "array_item_desc": "một dbSNP rsID hợp lệ, ví dụ rs104894090"},
    {"method": "GET", "path": "/annotation/dbsnp/{variantId}", "path_params": {"variantId": "một dbSNP rsID hợp lệ, ví dụ rs104894090"}, "body_mode": "none"},
    {"method": "POST", "path": "/annotation/genomic", "path_params": {}, "body_mode": "array", "array_item_desc": "một vị trí genomic dạng comma-separated (chromosome,start,end,ref,alt), ví dụ 7,140453136,140453136,A,T"},
    {"method": "GET", "path": "/annotation/genomic/{genomicLocation}", "path_params": {"genomicLocation": "vị trí genomic dạng comma-separated (chromosome,start,end,ref,alt), ví dụ 7,140453136,140453136,A,T"}, "body_mode": "none"},

    # --- ensembl-controller ---
    {"method": "POST", "path": "/ensembl/canonical-gene/entrez", "path_params": {}, "body_mode": "array", "array_item_desc": "một Entrez Gene ID hợp lệ (dạng số), ví dụ 672"},
    {"method": "GET", "path": "/ensembl/canonical-gene/entrez/{entrezGeneId}", "path_params": {"entrezGeneId": "một Entrez Gene ID hợp lệ (dạng số), ví dụ 672"}, "body_mode": "none"},
    {"method": "POST", "path": "/ensembl/canonical-gene/hgnc", "path_params": {}, "body_mode": "array", "array_item_desc": "một mã gen HGNC/HUGO hợp lệ, ví dụ BRCA1"},
    {"method": "GET", "path": "/ensembl/canonical-gene/hgnc/{hugoSymbol}", "path_params": {"hugoSymbol": "một mã gen HGNC/HUGO hợp lệ, ví dụ BRCA1"}, "body_mode": "none"},
    {"method": "POST", "path": "/ensembl/canonical-transcript/hgnc", "path_params": {}, "body_mode": "array", "array_item_desc": "một mã gen HGNC/HUGO hợp lệ, ví dụ BRCA1"},
    {"method": "GET", "path": "/ensembl/canonical-transcript/hgnc/{hugoSymbol}", "path_params": {"hugoSymbol": "một mã gen HGNC/HUGO hợp lệ, ví dụ BRCA1"}, "body_mode": "none"},
    {"method": "GET", "path": "/ensembl/transcript", "path_params": {}, "body_mode": "none"},
    {"method": "POST", "path": "/ensembl/transcript", "path_params": {}, "body_mode": "array", "array_item_desc": "một Ensembl transcript ID hợp lệ, ví dụ ENST00000269571"},
    {"method": "GET", "path": "/ensembl/transcript/{transcriptId}", "path_params": {"transcriptId": "một Ensembl transcript ID hợp lệ, ví dụ ENST00000269571"}, "body_mode": "none"},
    {"method": "GET", "path": "/ensembl/xrefs", "path_params": {}, "body_mode": "none"},

    # --- info-controller ---
    {"method": "GET", "path": "/version", "path_params": {}, "body_mode": "none"},

    # --- pdb-controller ---
    {"method": "POST", "path": "/pdb/header", "path_params": {}, "body_mode": "array", "array_item_desc": "một mã PDB ID hợp lệ, ví dụ 2ERK"},
    {"method": "GET", "path": "/pdb/header/{pdbId}", "path_params": {"pdbId": "một mã PDB ID hợp lệ, ví dụ 2ERK"}, "body_mode": "none"},

    # --- pfam-controller ---
    {"method": "POST", "path": "/pfam/domain", "path_params": {}, "body_mode": "array", "array_item_desc": "một mã Pfam accession hợp lệ, ví dụ PF00069"},
    {"method": "GET", "path": "/pfam/domain/{pfamAccession}", "path_params": {"pfamAccession": "một mã Pfam accession hợp lệ, ví dụ PF00069"}, "body_mode": "none"},

    # --- ptm-controller (suy ra theo pattern, controller bị thu gọn trong swagger) ---
    {"method": "POST", "path": "/ptm", "path_params": {}, "body_mode": "array", "array_item_desc": "một biến thể HGVS hợp lệ, ví dụ 7:g.140453136A>T"},
    {"method": "GET", "path": "/ptm/{variant}", "path_params": {"variant": "một biến thể HGVS hợp lệ, ví dụ 7:g.140453136A>T"}, "body_mode": "none"},
]

def get_llm_response(prompt):
    resp = client.models.generate_content(model=TARGET_MODEL, contents=prompt)
    return resp.text

def extract_answer(raw):
    m = re.search(r"<answer>(.*?)(?:</answer>|<answer>|$)", raw, re.S | re.I)
    return m.group(1).strip() if m else raw.strip()

def gen_value(api_path, pname, pdesc, error_feedback=None):
    if error_feedback:
        prompt = f"""You are an API tester. API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Previous attempt failed with error: {error_feedback}
Give me a corrected valid value. Output ONLY <answer>your answer<answer>"""
    else:
        prompt = f"""You are an API tester. Generate a valid value for the parameter below.
API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Output ONLY <answer>your answer<answer>"""
    return extract_answer(get_llm_response(prompt))

def build_call(ep, feedback=None):
    path = ep["path"]
    for pname, pdesc in ep.get("path_params", {}).items():
        fb = feedback.get(pname) if feedback else None
        val = gen_value(ep["path"], pname, pdesc, fb)
        path = path.replace("{" + pname + "}", str(val))

    body_mode = ep.get("body_mode", "none")
    if body_mode == "array":
        fb = feedback.get("_array_item") if feedback else None
        val = gen_value(ep["path"], "array item", ep["array_item_desc"], fb)
        body = [val]
    elif body_mode == "object":
        body = {}
        for pname, pdesc in ep.get("body_params", {}).items():
            fb = feedback.get(pname) if feedback else None
            body[pname] = gen_value(ep["path"], pname, pdesc, fb)
    else:
        body = None
    return path, body

def run_endpoint(ep):
    path_a, body_a = build_call(ep)
    try:
        resp_a = requests.request(ep["method"], BASE_URL + path_a, json=body_a, timeout=10)
        status_a, err_text = resp_a.status_code, resp_a.text[:300]
    except Exception as e:
        status_a, err_text = -1, str(e)

    status_b = status_a
    if status_a >= 400 or status_a == -1:
        feedback = {k: err_text for k in ep.get("path_params", {})}
        if ep.get("body_mode") == "array":
            feedback["_array_item"] = err_text
        elif ep.get("body_mode") == "object":
            for k in ep.get("body_params", {}):
                feedback[k] = err_text
        path_b, body_b = build_call(ep, feedback)
        try:
            resp_b = requests.request(ep["method"], BASE_URL + path_b, json=body_b, timeout=10)
            status_b = resp_b.status_code
        except Exception:
            status_b = -1
    return status_a, status_b

results = []
for ep in ENDPOINTS:
    print(f">>> Test {ep['method']} {ep['path']}")
    sa, sb = run_endpoint(ep)
    print(f"    w/o feedback={sa}  w/ feedback={sb}")
    results.append({"service": SERVICE_NAME, "endpoint": ep["path"], "status_wo_feedback": sa, "status_w_feedback": sb})

df = pd.DataFrame(results)
def is2xx(s): return isinstance(s, int) and 200 <= s < 300
def is5xx(s): return isinstance(s, int) and 500 <= s < 600

summary = {
    "service": SERVICE_NAME,
    "2XX_wo_feedback": df["status_wo_feedback"].apply(is2xx).sum(),
    "2XX_w_feedback": df["status_w_feedback"].apply(is2xx).sum(),
    "5XX_wo_feedback": df["status_wo_feedback"].apply(is5xx).sum(),
    "5XX_w_feedback": df["status_w_feedback"].apply(is5xx).sum(),
}
print("\n[SUMMARY]", summary)

os.makedirs("/content/drive/MyDrive/RBL-experiment/results", exist_ok=True)
df.to_csv(f"/content/drive/MyDrive/RBL-experiment/results/{SERVICE_NAME}_results.csv", index=False)
print(f"\n[FINISH] Đã test xong {SERVICE_NAME}!")

>>> Test GET /annotation/{variant}
    w/o feedback=200  w/ feedback=200
>>> Test POST /annotation
    w/o feedback=200  w/ feedback=200
>>> Test POST /annotation/dbsnp/
    w/o feedback=200  w/ feedback=200
>>> Test GET /annotation/dbsnp/{variantId}
    w/o feedback=200  w/ feedback=200
>>> Test POST /annotation/genomic
    w/o feedback=405  w/ feedback=405
>>> Test GET /annotation/genomic/{genomicLocation}
    w/o feedback=200  w/ feedback=200
>>> Test POST /ensembl/canonical-gene/entrez
    w/o feedback=200  w/ feedback=200
>>> Test GET /ensembl/canonical-gene/entrez/{entrezGeneId}
    w/o feedback=200  w/ feedback=200
>>> Test POST /ensembl/canonical-gene/hgnc
    w/o feedback=200  w/ feedback=200
>>> Test GET /ensembl/canonical-gene/hgnc/{hugoSymbol}
    w/o feedback=200  w/ feedback=200
>>> Test POST /ensembl/canonical-transcript/hgnc
    w/o feedback=200  w/ feedback=200
>>> Test GET /ensembl/canonical-transcript/hgnc/{hugoSymbol}
    w/o feedback=200  w/ feedback=200
>>> Test G

In [ ]:
# [CELL: realworld - LLM-only vs LLM+Feedback]
import os, re, json, requests, pandas as pd
from google import genai

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"
client = genai.Client(vertexai=True, project="rbl-core-2026", location="asia-southeast1")
TARGET_MODEL = "gemini-3.5-flash"
BASE_URL = "https://api.realworld.show"
SERVICE_NAME = "realworld"

# Lưu ý: dùng demo public chính thức theo chuẩn RealWorld/Conduit spec
# (KHÔNG phải đúng 100% backend nestjs-realworld-example-app trong bài báo, vì backend đó không có public demo)
ENDPOINTS = [
    # --- ArticleController (base: /articles) ---
    {"method": "GET", "path": "/api/articles", "path_params": {}, "body_params": {}, "wrapper": None},
    {"method": "GET", "path": "/api/articles/feed", "path_params": {}, "body_params": {}, "wrapper": None},
    {"method": "GET", "path": "/api/articles/{slug}", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {}, "wrapper": None},
    {"method": "GET", "path": "/api/articles/{slug}/comments", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {}, "wrapper": None},
    {"method": "POST", "path": "/api/articles", "path_params": {}, "body_params": {
        "title": "tiêu đề bài viết hợp lệ",
        "description": "mô tả ngắn cho bài viết",
        "body": "nội dung chính của bài viết",
    }, "wrapper": "article"},
    {"method": "PUT", "path": "/api/articles/{slug}", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {
        "title": "tiêu đề mới cho bài viết",
    }, "wrapper": "article"},
    {"method": "DELETE", "path": "/api/articles/{slug}", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {}, "wrapper": None},
    {"method": "POST", "path": "/api/articles/{slug}/comments", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {
        "body": "nội dung bình luận hợp lệ",
    }, "wrapper": "comment"},
    {"method": "DELETE", "path": "/api/articles/{slug}/comments/{id}", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon", "id": "một ID bình luận hợp lệ (dạng số)"}, "body_params": {}, "wrapper": None},
    {"method": "POST", "path": "/api/articles/{slug}/favorite", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {}, "wrapper": None},
    {"method": "DELETE", "path": "/api/articles/{slug}/favorite", "path_params": {"slug": "một slug bài viết hợp lệ, ví dụ how-to-train-your-dragon"}, "body_params": {}, "wrapper": None},

    # --- UserController (base: /user, /users) ---
    {"method": "GET", "path": "/api/user", "path_params": {}, "body_params": {}, "wrapper": None},
    {"method": "PUT", "path": "/api/user", "path_params": {}, "body_params": {
        "bio": "một đoạn tiểu sử ngắn hợp lệ",
    }, "wrapper": "user"},
    {"method": "POST", "path": "/api/users", "path_params": {}, "body_params": {
        "username": "một username hợp lệ, duy nhất",
        "email": "một địa chỉ email hợp lệ",
        "password": "một mật khẩu hợp lệ, ít nhất 8 ký tự",
    }, "wrapper": "user"},
    {"method": "DELETE", "path": "/api/users/{slug}", "path_params": {"slug": "một username hợp lệ đã tồn tại"}, "body_params": {}, "wrapper": None},
    {"method": "POST", "path": "/api/users/login", "path_params": {}, "body_params": {
        "email": "một địa chỉ email hợp lệ đã đăng ký",
        "password": "mật khẩu tương ứng với email đó",
    }, "wrapper": "user"},

    # --- ProfileController (base: /profiles) ---
    {"method": "GET", "path": "/api/profiles/{username}", "path_params": {"username": "một username hợp lệ đã tồn tại"}, "body_params": {}, "wrapper": None},
    {"method": "POST", "path": "/api/profiles/{username}/follow", "path_params": {"username": "một username hợp lệ đã tồn tại"}, "body_params": {}, "wrapper": None},
    {"method": "DELETE", "path": "/api/profiles/{username}/follow", "path_params": {"username": "một username hợp lệ đã tồn tại"}, "body_params": {}, "wrapper": None},
]

def get_llm_response(prompt):
    resp = client.models.generate_content(model=TARGET_MODEL, contents=prompt)
    return resp.text

def extract_answer(raw):
    m = re.search(r"<answer>(.*?)(?:</answer>|<answer>|$)", raw, re.S | re.I)
    return m.group(1).strip() if m else raw.strip()

def gen_value(api_path, pname, pdesc, error_feedback=None):
    if error_feedback:
        prompt = f"""You are an API tester. API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Previous attempt failed with error: {error_feedback}
Give me a corrected valid value. Output ONLY <answer>your answer<answer>"""
    else:
        prompt = f"""You are an API tester. Generate a valid value for the parameter below.
API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Output ONLY <answer>your answer<answer>"""
    return extract_answer(get_llm_response(prompt))

def build_call(ep, feedback=None):
    path = ep["path"]
    body = {}
    for pname, pdesc in ep["path_params"].items():
        fb = feedback.get(pname) if feedback else None
        val = gen_value(ep["path"], pname, pdesc, fb)
        path = path.replace("{" + pname + "}", str(val))
    for pname, pdesc in ep["body_params"].items():
        fb = feedback.get(pname) if feedback else None
        body[pname] = gen_value(ep["path"], pname, pdesc, fb)
    final_body = {ep["wrapper"]: body} if ep["wrapper"] and body else (body or None)
    return path, final_body

def run_endpoint(ep):
    path_a, body_a = build_call(ep)
    try:
        resp_a = requests.request(ep["method"], BASE_URL + path_a, json=body_a, timeout=10)
        status_a, err_text = resp_a.status_code, resp_a.text[:300]
    except Exception as e:
        status_a, err_text = -1, str(e)

    status_b = status_a
    if status_a >= 400 or status_a == -1:
        feedback = {k: err_text for k in list(ep["path_params"]) + list(ep["body_params"])}
        path_b, body_b = build_call(ep, feedback)
        try:
            resp_b = requests.request(ep["method"], BASE_URL + path_b, json=body_b, timeout=10)
            status_b = resp_b.status_code
        except Exception:
            status_b = -1
    return status_a, status_b

results = []
for ep in ENDPOINTS:
    print(f">>> Test {ep['method']} {ep['path']}")
    sa, sb = run_endpoint(ep)
    print(f"    w/o feedback={sa}  w/ feedback={sb}")
    results.append({"service": SERVICE_NAME, "endpoint": ep["path"], "status_wo_feedback": sa, "status_w_feedback": sb})

df = pd.DataFrame(results)
def is2xx(s): return isinstance(s, int) and 200 <= s < 300
def is5xx(s): return isinstance(s, int) and 500 <= s < 600

summary = {
    "service": SERVICE_NAME,
    "2XX_wo_feedback": df["status_wo_feedback"].apply(is2xx).sum(),
    "2XX_w_feedback": df["status_w_feedback"].apply(is2xx).sum(),
    "5XX_wo_feedback": df["status_wo_feedback"].apply(is5xx).sum(),
    "5XX_w_feedback": df["status_w_feedback"].apply(is5xx).sum(),
}
print("\n[SUMMARY]", summary)

os.makedirs("/content/drive/MyDrive/RBL-experiment/results", exist_ok=True)
df.to_csv(f"/content/drive/MyDrive/RBL-experiment/results/{SERVICE_NAME}_results.csv", index=False)
print(f"\n[FINISH] Đã test xong {SERVICE_NAME}!")

>>> Test GET /api/articles
    w/o feedback=200  w/ feedback=200
>>> Test GET /api/articles/feed
    w/o feedback=401  w/ feedback=401
>>> Test GET /api/articles/{slug}
    w/o feedback=404  w/ feedback=404
>>> Test GET /api/articles/{slug}/comments
    w/o feedback=404  w/ feedback=404
>>> Test POST /api/articles
    w/o feedback=401  w/ feedback=401
>>> Test PUT /api/articles/{slug}
    w/o feedback=401  w/ feedback=401
>>> Test DELETE /api/articles/{slug}
    w/o feedback=401  w/ feedback=401
>>> Test POST /api/articles/{slug}/comments
    w/o feedback=401  w/ feedback=401
>>> Test DELETE /api/articles/{slug}/comments/{id}
    w/o feedback=401  w/ feedback=401
>>> Test POST /api/articles/{slug}/favorite
    w/o feedback=401  w/ feedback=401
>>> Test DELETE /api/articles/{slug}/favorite
    w/o feedback=401  w/ feedback=401
>>> Test GET /api/user
    w/o feedback=401  w/ feedback=401
>>> Test PUT /api/user
    w/o feedback=401  w/ feedback=401
>>> Test POST /api/users
    w/o feedbac

In [ ]:
# [CELL: GitLab - tải + parse OpenAPI spec thật, lấy đúng endpoint]
import requests, yaml

OPENAPI_URL = "https://gitlab.com/gitlab-org/gitlab/-/raw/master/doc/api/openapi/openapi.yaml"
resp = requests.get(OPENAPI_URL, timeout=30)
spec = yaml.safe_load(resp.text)

paths = spec.get("paths", {})
print(f"Tổng số path trong spec gốc: {len(paths)}")

all_endpoints = []
for path, methods in paths.items():
    for method, detail in methods.items():
        if method.lower() not in ("get", "post", "put", "delete", "patch"):
            continue
        path_params = {}
        for p in detail.get("parameters", []):
            if p.get("in") == "path":
                path_params[p["name"]] = p.get("description", f"giá trị hợp lệ cho {p['name']}")
        body_params = {}
        rb = detail.get("requestBody", {})
        json_schema = rb.get("content", {}).get("application/json", {}).get("schema", {})
        for pname, pinfo in json_schema.get("properties", {}).items():
            body_params[pname] = pinfo.get("description", f"giá trị hợp lệ cho {pname}")
        all_endpoints.append({
            "method": method.upper(),
            "path": path,
            "path_params": path_params,
            "body_params": body_params,
        })

print(f"Tổng số endpoint (method+path) trong spec: {len(all_endpoints)}")

# Lấy đúng 99 endpoint đầu tiên (theo thứ tự trong spec) để khớp Table 2 của bài báo
ENDPOINTS = all_endpoints[:99]
print(f"Đã chọn {len(ENDPOINTS)} endpoint để test.")
for ep in ENDPOINTS[:5]:
    print(" ", ep["method"], ep["path"])

ScannerError: mapping values are not allowed here
  in "<unicode string>", line 9, column 45:
     ... atchMedia('(prefers-color-scheme: dark)').matches) {
                                         ^

In [ ]:
# [CELL: GitLab - LLM-only vs LLM+Feedback]
import os, re, json, requests, pandas as pd
from google import genai

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/new-gcp-key.json"
client = genai.Client(vertexai=True, project="rbl-core-2026", location="asia-southeast1")
TARGET_MODEL = "gemini-3.5-flash"
BASE_URL = "https://gitlab.com/api/v4"
SERVICE_NAME = "GitLab"

# Chỉ test các endpoint public, không cần GITLAB_TOKEN
ENDPOINTS = [
    # ---------- PROJECTS ----------
    {"method": "GET", "path": "/projects/{id}", "path_params": {"id": "id dạng URL-encoded của một project GitLab public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/users", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/members", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/members/all", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/members/{user_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "user_id": "id số của một thành viên trong project, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/languages", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/forks", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/starrers", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/events", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/hooks", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/branches", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/branches/{branch}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "branch": "tên branch tồn tại trong project, ví dụ master hoặc main"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/commits", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/commits/{sha}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "sha": "SHA của một commit hợp lệ (chuỗi hex 40 ký tự) hoặc tên branch"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/commits/{sha}/diff", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "sha": "SHA của một commit hợp lệ hoặc tên branch"}, "body_params": {}},
{"method": "GET", "path": "/projects/{id}/repository/commits/{sha}/comments", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "sha": "SHA của một commit hợp lệ hoặc tên branch"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/commits/{sha}/refs", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "sha": "SHA của một commit hợp lệ"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/commits/{sha}/statuses", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "sha": "SHA của một commit hợp lệ"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/tags", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/tags/{tag_name}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "tag_name": "tên một tag tồn tại trong project, ví dụ v1.0.0"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/tree", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/contributors", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/repository/files/{file_path}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "file_path": "đường dẫn file dạng URL-encoded trong repo, ví dụ README.md"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/issues", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/issues/{issue_iid}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "issue_iid": "IID (số nội bộ project) của một issue, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/issues/{issue_iid}/notes", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "issue_iid": "IID của một issue, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/issues/{issue_iid}/award_emoji", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "issue_iid": "IID của một issue, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/issues/{issue_iid}/closed_by", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "issue_iid": "IID của một issue, ví dụ 1"}, "body_params": {}},
{"method": "GET", "path": "/projects/{id}/issues/{issue_iid}/related_merge_requests", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "issue_iid": "IID của một issue, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/issues/{issue_iid}/participants", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "issue_iid": "IID của một issue, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}/commits", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}/changes", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}/notes", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}/participants", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}/approvals", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/merge_requests/{merge_request_iid}/award_emoji", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "merge_request_iid": "IID của một merge request, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/milestones", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/milestones/{milestone_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "milestone_id": "id số của một milestone trong project, ví dụ 1"}, "body_params": {}},
{"method": "GET", "path": "/projects/{id}/labels", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/pipelines", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/pipelines/{pipeline_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "pipeline_id": "id số của một pipeline trong project, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/pipelines/{pipeline_id}/jobs", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "pipeline_id": "id số của một pipeline, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/pipelines/{pipeline_id}/variables", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "pipeline_id": "id số của một pipeline, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/pipelines/{pipeline_id}/test_report", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "pipeline_id": "id số của một pipeline, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/jobs", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/jobs/{job_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "job_id": "id số của một job trong project, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/jobs/{job_id}/trace", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "job_id": "id số của một job, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/releases", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/releases/{tag_name}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "tag_name": "tên một tag có release, ví dụ v1.0.0"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/environments", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/environments/{environment_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "environment_id": "id số của một environment, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/deployments", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
{"method": "GET", "path": "/projects/{id}/snippets", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/snippets/{snippet_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "snippet_id": "id số của một snippet trong project, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/variables", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/registry/repositories", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/packages", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/wikis", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/wikis/{slug}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "slug": "slug của một trang wiki, ví dụ home"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/boards", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/boards/{board_id}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "board_id": "id số của một board, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/protected_branches", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/protected_branches/{name}", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab", "name": "tên một protected branch, ví dụ main"}, "body_params": {}},
    {"method": "GET", "path": "/projects/{id}/access_requests", "path_params": {"id": "id URL-encoded của project public, ví dụ gitlab-org%2Fgitlab"}, "body_params": {}},

    # ---------- GROUPS ----------
    {"method": "GET", "path": "/groups/{id}", "path_params": {"id": "id URL-encoded hoặc path của một group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/subgroups", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/descendant_groups", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
{"method": "GET", "path": "/groups/{id}/projects", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/projects/shared", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/members", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/members/all", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/members/{user_id}", "path_params": {"id": "id/path của group public, ví dụ gitlab-org", "user_id": "id số của một thành viên trong group, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/issues", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/merge_requests", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/milestones", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/milestones/{milestone_id}", "path_params": {"id": "id/path của group public, ví dụ gitlab-org", "milestone_id": "id số của một milestone trong group, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/labels", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/epics", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/epics/{epic_iid}", "path_params": {"id": "id/path của group public, ví dụ gitlab-org", "epic_iid": "IID của một epic trong group, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/variables", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/boards", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/badges", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/access_requests", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/groups/{id}/hooks", "path_params": {"id": "id/path của group public, ví dụ gitlab-org"}, "body_params": {}},

    # ---------- USERS ----------
    {"method": "GET", "path": "/users/{id}", "path_params": {"id": "id số của một user GitLab public, ví dụ 1"}, "body_params": {}},
{"method": "GET", "path": "/users/{id}/projects", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/followers", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/following", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/keys", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/keys/{key_id}", "path_params": {"id": "id số của một user public, ví dụ 1", "key_id": "id số của một SSH key của user, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/gpg_keys", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/status", "path_params": {"id": "id số hoặc username của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/events", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/users/{id}/starred_projects", "path_params": {"id": "id số của một user public, ví dụ 1"}, "body_params": {}},

    # ---------- OTHER ----------
    {"method": "GET", "path": "/namespaces/{id}", "path_params": {"id": "id số hoặc path của một namespace public, ví dụ gitlab-org"}, "body_params": {}},
    {"method": "GET", "path": "/snippets/{id}", "path_params": {"id": "id số của một public snippet, ví dụ 1"}, "body_params": {}},
    {"method": "GET", "path": "/snippets/{id}/raw", "path_params": {"id": "id số của một public snippet, ví dụ 1"}, "body_params": {}},
]

def get_llm_response(prompt):
    resp = client.models.generate_content(model=TARGET_MODEL, contents=prompt)
    return resp.text

def extract_answer(raw):
    m = re.search(r"<answer>(.*?)(?:</answer>|<answer>|$)", raw, re.S | re.I)
    return m.group(1).strip() if m else raw.strip()

def gen_value(api_path, pname, pdesc, error_feedback=None):
    if error_feedback:
        prompt = f"""You are an API tester. API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Previous attempt failed with error: {error_feedback}
Give me a corrected valid value. Output ONLY <answer>your answer<answer>"""
    else:
        prompt = f"""You are an API tester. Generate a valid value for the parameter below.
API: {api_path}
Parameter name: {pname}. Description: {pdesc}
Output ONLY <answer>your answer<answer>"""
    return extract_answer(get_llm_response(prompt))

def build_call(ep, feedback=None):
    path = ep["path"]
    for pname, pdesc in ep["path_params"].items():
        fb = feedback.get(pname) if feedback else None
        val = gen_value(ep["path"], pname, pdesc, fb)
        path = path.replace("{" + pname + "}", str(val))
    return path

def run_endpoint(ep):
    path_a = build_call(ep)
    try:
        resp_a = requests.get(BASE_URL + path_a, timeout=10)
        status_a, err_text = resp_a.status_code, resp_a.text[:300]
    except Exception as e:
        status_a, err_text = -1, str(e)

    status_b = status_a
    if status_a >= 400 or status_a == -1:
        feedback = {k: err_text for k in ep["path_params"]}
        path_b = build_call(ep, feedback)
        try:
            resp_b = requests.get(BASE_URL + path_b, timeout=10)
            status_b = resp_b.status_code
        except Exception:
            status_b = -1
    return status_a, status_b

results = []
for ep in ENDPOINTS:
    print(f">>> Test {ep['method']} {ep['path']}")
    sa, sb = run_endpoint(ep)
    print(f"    w/o feedback={sa}  w/ feedback={sb}")
    results.append({"service": SERVICE_NAME, "endpoint": ep["path"], "status_wo_feedback": sa, "status_w_feedback": sb})

df = pd.DataFrame(results)
def is2xx(s): return isinstance(s, int) and 200 <= s < 300
def is5xx(s): return isinstance(s, int) and 500 <= s < 600

summary = {
    "service": SERVICE_NAME,
    "2XX_wo_feedback": df["status_wo_feedback"].apply(is2xx).sum(),
    "2XX_w_feedback": df["status_w_feedback"].apply(is2xx).sum(),
    "5XX_wo_feedback": df["status_wo_feedback"].apply(is5xx).sum(),
    "5XX_w_feedback": df["status_w_feedback"].apply(is5xx).sum(),
}
print("\n[SUMMARY]", summary)

os.makedirs("/content/drive/MyDrive/RBL-experiment/results", exist_ok=True)
df.to_csv(f"/content/drive/MyDrive/RBL-experiment/results/{SERVICE_NAME}_results.csv", index=False)
print(f"\n[FINISH] Đã test xong {SERVICE_NAME}!")

>>> Test GET /projects/{id}


DefaultCredentialsError: File /content/new-gcp-key.json was not found.

In [ ]:
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = "GiaPhu-1511"          # chủ sở hữu repo
REPO_NAME = "RT-SWT-005-nhom6"
BRANCH_NAME = "Flower"

repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
os.chdir("/content")

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {repo_url}

os.chdir(f"/content/{REPO_NAME}")
!git config user.email "nguyenhoa.nt0513@gmail.com"   # sửa thành email gắn với tài khoản Mimosa-flower
!git config user.name "Mimosa-flower"

# Kiểm tra nhánh Flower đã tồn tại trên remote chưa
!git fetch origin
branch_exists = os.system(f"git show-ref --verify --quiet refs/remotes/origin/{BRANCH_NAME}") == 0

if branch_exists:
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}
else:
    !git checkout -b {BRANCH_NAME}

!git branch

Cloning into 'RT-SWT-005-nhom6'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 85 (delta 0), reused 0 (delta 0), pack-reused 84 (from 1)
Receiving objects: 100% (85/85), 35.45 MiB | 25.57 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Branch 'Flower' set up to track remote branch 'Flower' from 'origin'.
Switched to a new branch 'Flower'
From https://github.com/GiaPhu-1511/RT-SWT-005-nhom6
 * branch            Flower     -> FETCH_HEAD
Already up to date.
* Flower
  main


In [ ]:
import shutil, os

os.chdir(f"/content/{REPO_NAME}")
os.makedirs("results", exist_ok=True)

src_dir = "/content/drive/MyDrive/RBL-experiment/results"
for fname in os.listdir(src_dir):
    if fname.endswith(".csv") or fname.endswith(".txt"):
        shutil.copy(os.path.join(src_dir, fname), f"results/{fname}")

!git add results/
!git status

On branch Flower
Your branch is up to date with 'origin/Flower'.

nothing to commit, working tree clean


In [ ]:
!git commit -m "feat: add pilot experiment results (5 services nhom A, LLM vs LLM+feedback)"
!git push origin Flower

[Flower b3e39cc] feat: add pilot experiment results (5 services nhom A, LLM vs LLM+feedback)
 8 files changed, 48 insertions(+)
 create mode 100644 results/GitLab_results.csv
 create mode 100644 results/comparison_results.csv
 create mode 100644 results/disease.sh_results.csv
 create mode 100644 results/genome-nexus_results.csv
 create mode 100644 results/languagetool_results.csv
 create mode 100644 results/pilot_llm_output_checkpoint.csv
 create mode 100644 results/realworld_results.csv
Enumerating objects: 14, done.
Counting objects: 100% (14/14), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (11/11), 2.14 KiB | 437.00 KiB/s, done.
Total 11 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/GiaPhu-1511/RT-SWT-005-nhom6.git
   8633467..b3e39cc  Flower -> Flower
